# 不使用@tool的方式定义工具

## 1、举例

In [1]:
# 1、模型的初始化
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

from langchain_core.utils.function_calling import convert_to_openai_tool
from rich import print as rprint

# 从.env文件中加载环境变量
load_dotenv(override=True)

model = init_chat_model(
    model="deepseek-v4-pro",
    model_provider="openai",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL")
)

# 2、声明一个函数（工具）
def get_weather(city : str):
    return f"{city}天气晴朗~~"

# 3、将函数绑定在模型上
model_with_tools = model.bind_tools([get_weather])

# 4、调用模型
response = model_with_tools.invoke("北京的天气怎么样")
rprint(response)

AIMessage(
    content='我来帮您查询北京的天气。',
    additional_kwargs={'refusal': None},
    response_metadata={
        'token_usage': {
            'completion_tokens': 77,
            'prompt_tokens': 269,
            'total_tokens': 346,
            'completion_tokens_details': {
                'accepted_prediction_tokens': None,
                'audio_tokens': None,
                'reasoning_tokens': 26,
                'rejected_prediction_tokens': None
            },
            'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0},
            'prompt_cache_hit_tokens': 0,
            'prompt_cache_miss_tokens': 269
        },
        'model_provider': 'openai',
        'model_name': 'deepseek-v4-pro',
        'system_fingerprint': 'fp_9954b31ca7_prod0820_fp8_kvcache_20260402',
        'id': '23bbbb05-c55f-4cb0-ba07-e4871197317f',
        'finish_reason': 'tool_calls',
        'logprobs': None
    },
    id='lc_run--019f0843-f2b5-7451-85d3-722a2ca42092-0',
    tool_calls=[
        {
            'name': 'get_weather',
            'args': {'city': '北京'},
            'id': 'call_00_tFIU9DiRAElPlP0hmoAN8791',
            'type': 'tool_call'
        }
    ],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 269,
        'output_tokens': 77,
        'total_tokens': 346,
        'input_token_details': {'cache_read': 0},
        'output_token_details': {'reasoning': 26}
    }
)

## 2、工具描述的各部分详解

## 2.1 了解convert_to_openai_tool

执行`model.bind_tools([get_weather])`，底层最终会调用`convert_to_openai_tool`生成工具描述。所以我们可以直接调用后者查看解析后的工具描述。

In [2]:
from langchain_core.utils.function_calling import convert_to_openai_tool

def get_weather(city : str):
    return f"{city}天气晴朗~~"


rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '',
        'parameters': {'properties': {'city': {'type': 'string'}}, 'required': ['city'], 'type': 'object'}
    }
}

## 2.2 description说明

In [3]:
from langchain_core.utils.function_calling import convert_to_openai_tool

def get_weather(city : str):
    """
    查询城市的天气
    """
    return f"{city}天气晴朗~~"


rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '查询城市的天气',
        'parameters': {'properties': {'city': {'type': 'string'}}, 'required': ['city'], 'type': 'object'}
    }
}

## 2.3 参数说明

In [4]:
from langchain_core.utils.function_calling import convert_to_openai_tool

def get_weather(city : str):
    """
    查询城市的天气

    Args:
        city : 具体的城市

    Returns:
        返回城市的天气
    """
    return f"{city}天气晴朗~~"


rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '查询城市的天气',
        'parameters': {
            'properties': {'city': {'description': '具体的城市', 'type': 'string'}},
            'required': ['city'],
            'type': 'object'
        }
    }
}

## 2.4 参数类型说明

举例1：正确的

In [12]:
from langchain_core.utils.function_calling import convert_to_openai_tool

def get_weather(city):
    """
    查询城市的天气
    """
    return f"{city}天气晴朗~~"


rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '查询城市的天气',
        'parameters': {'properties': {'city': {}}, 'required': ['city'], 'type': 'object'}
    }
}

举例2：如下的代码运行会报错

要求：如果在docstring中声明了参数的描述，则必须在函数声明处指明参数的类型

In [5]:
from langchain_core.utils.function_calling import convert_to_openai_tool

def get_weather(city):
    """
    查询城市的天气

    Args:
        city : 具体的城市
    """
    return f"{city}天气晴朗~~"


rprint(convert_to_openai_tool(get_weather))

ValueError: Arg city in docstring not found in function signature.

## 2.5 参数默认值说明

一旦参数设置的默认值，则打印的结果中的required字段中就不再包含此参数。

举例1：

In [6]:
from langchain_core.utils.function_calling import convert_to_openai_tool

def get_weather(city : str = "beijing"):
    """
    查询城市的天气

    Args:
        city : 具体的城市
    """
    return f"{city}天气晴朗~~"


rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '查询城市的天气',
        'parameters': {
            'properties': {'city': {'default': 'beijing', 'description': '具体的城市', 'type': 'string'}},
            'type': 'object'
        }
    }
}

举例2：

In [15]:
from langchain_core.utils.function_calling import convert_to_openai_tool

def get_weather(dt:str ,city : str = "beijing"):
    """
    查询城市的天气

    Args:
        city : 具体的城市
        dt : 时间
    """
    return f"{city}天气晴朗~~"


rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '查询城市的天气',
        'parameters': {
            'properties': {
                'dt': {'description': '时间', 'type': 'string'},
                'city': {'default': 'beijing', 'description': '具体的城市', 'type': 'string'}
            },
            'required': ['dt'],
            'type': 'object'
        }
    }
}